# LangChain의 `create_agent`에 대해 학습

In [ ]:
from langchain.agents import create_agent 
from langchain.agents.middleware import wrap_tool_call
from langchain.tools import tool

import logging

logging.basicConfig(
      level=logging.DEBUG,
      format="%(asctime)s %(levelname)s %(name)s - %(message)s",
  )

logger = logging.getLogger(__name__)

# 1. 도구 정의
@tool
def get_weather(location: str) -> str:
    """Get weather information for a specific location.

    Args:
        location: The city or location name (e.g., "Seoul", "New York")

    Returns:
        Weather information as a string
    """
    # 실제로는 날씨 API를 호출하지만, 여기서는 예시로 고정값 반환
    weather_data = {
        "서울": "맑음, 25°C, 습도 60%",
        "뉴욕": "흐림, 18°C, 습도 75%",
        "도쿄": "비, 20°C, 습도 85%"
    }
    return weather_data.get(location, f"{location}의 날씨 정보를 찾을 수 없습니다.")

@wrap_tool_call
def log_tool_call(request, handler):
    """Log every tool call before and after execution."""
    tool_name = request.tool_call["name"]
    args = request.tool_call["args"]

    logger.debug("Tool start: %s, Args: %s", tool_name, args)
    result = handler(request)
    result_content = getattr(result, "content", result)
    logger.debug("Tool end: %s, Result: %s", tool_name, result_content)
    return result


# 2. 에이전트 생성
agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_weather],
    middleware=[log_tool_call],
    system_prompt="""You are a helpful weather assistant.
When users ask about weather, use the get_weather tool to fetch current information.
Always provide the weather information in a friendly and clear manner."""
)

# 3. 에이전트 실행
result = agent.invoke({
    "messages": [{"role": "user", "content": "서울 날씨 어때?"}]
})

print(result["messages"][-1].content)

2026-05-17 16:30:13,817 DEBUG openai._base_client - Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-b5e81070-e0fc-4210-9295-3ca3cc91fc3f', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are a helpful weather assistant.\nWhen users ask about weather, use the get_weather tool to fetch current information.\nAlways provide the weather information in a friendly and clear manner.', 'role': 'system'}, {'content': '서울 날씨 어때?', 'role': 'user'}], 'model': 'gpt-4o-mini', 'stream': False, 'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get weather information for a specific location.\n\nArgs:\n    location: The city or location name (e.g., "Seoul", "New York")\n\nReturns:\n    Weather information as a string', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location

서울의 현재 날씨는 맑고, 기온은 25°C입니다. 습도는 60%로 쾌적한 편이네요. 좋은 하루 보내세요!


2026-05-17 16:30:19,597 DEBUG langsmith.client - Sending compressed multipart request with context: trace=019e34d7-bab6-7752-a3ef-35788d6b464c,id=019e34d7-c901-7611-8170-ea64115fecaf; trace=019e34d7-bab6-7752-a3ef-35788d6b464c,id=019e34d7-c900-7b71-b6d4-e26f838cae46; trace=019e34d7-bab6-7752-a3ef-35788d6b464c,id=019e34d7-bab6-7752-a3ef-35788d6b464c
2026-05-17 16:30:19,801 DEBUG urllib3.connectionpool - https://api.smith.langchain.com:443 "POST /runs/multipart HTTP/1.1" 202 34


In [17]:
from langchain.chat_models import init_chat_model

def detect_harmful_content(text):
    """LLM을 사용한 모델 기반 유해 콘텐츠 감지"""
    llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

    prompt = f"""다음 텍스트가 유해한 콘텐츠를 포함하는지 판단하세요.
유해 콘텐츠는 욕설, 폭력, 차별, 혐오 표현 등을 포함합니다.

텍스트: {text}

JSON 형식으로 응답하세요:
{{"is_harmful": true/false, "reason": "판단 이유"}}"""

    response = llm.invoke(prompt)
    return response.content

# 테스트
user_input = "이 제품은 정말 그지같아요. 환불받고 싶습니다."
result = detect_harmful_content(user_input)
print(result)
# LLM이 맥락을 이해하여 단순 불만이지 유해 콘텐츠가 아님을 판단


{"is_harmful": true, "reason": "욕설이 포함되어 있어 유해한 콘텐츠로 판단됨"}
